
# Late Delivery Prediction — Machine Learning

## Project Overview

This project develops a **binary classification model** to predict whether an e-commerce order will arrive late using the Brazilian Olist e-commerce dataset.

The goal is to identify orders at risk of late delivery using information available **at the time of purchase**, helping businesses improve delivery reliability and customer satisfaction.

### Objectives
- Prepare and explore historical order data from PostgreSQL.
- Engineer predictive features while preventing data leakage.
- Train and compare Logistic Regression and Random Forest classifiers.
- Evaluate model performance using precision, recall, F1-score, and confusion matrices.

### Prediction Target
- **0 — Not Late:** Delivered on or before the estimated delivery date.
- **1 — Late:** Delivered after the estimated delivery date.

Orders with missing delivery outcomes will be excluded from training.


## 1. Data Extraction

Load historical order data from the PostgreSQL `retail_bi` database using Pandas and SQLAlchemy.

We'll begin with order-level information and our binary prediction target (`is_late`). Additional features will be introduced during feature engineering.

In [18]:
# Install the PostgreSQL driver into the notebook's Python environment, psycopg2 handles PostgreSQL communication
# %pip install psycopg2-binary

import os
import pandas as pd
# SQLAlchemy manages database connections
from sqlalchemy import create_engine, text
from getpass import getpass

# Prompt for the database password without displaying it in the notebook
db_password = getpass("PostgreSQL password: ")

# Create a connection to our local PostgreSQL database
engine = create_engine(
    "postgresql+psycopg2://",
    connect_args={
        "host": "localhost",
        "port": 5432,
        "dbname": "retail_bi",
        "user": "postgres",
        "password": db_password
    }
)

# Execute a simple SQL query to verify the database connection
with engine.connect() as connection:
    result = connection.execute(text("SELECT 1"))
    print("Connection successful:", result.scalar())

Connection successful: 1


In [19]:
# SQL query to retrieve historical orders and create our prediction target
query = """
SELECT
    order_id,
    customer_id,
    order_purchase_timestamp,
    order_estimated_delivery_date,
    delivery_status,

    -- Binary classification target: 1 = Late, 0 = Not Late
    CASE
        WHEN delivery_status = 'Late' THEN 1
        ELSE 0
    END AS is_late

FROM orders

-- Exclude orders with missing delivery outcomes
WHERE delivery_status IN ('Early', 'On Time', 'Late')
"""

# Execute the SQL query and load the results into a Pandas DataFrame
df = pd.read_sql(query, engine)

# Preview the first five rows
print(df.head())

# Display the number of rows and columns in our dataset
print("Dataset shape:", df.shape)

# Count how many orders belong to each target class
print("\nTarget distribution:")
print(df["is_late"].value_counts())

                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

  order_purchase_timestamp order_estimated_delivery_date delivery_status  \
0      2017-10-02 10:56:33                    2017-10-18           Early   
1      2018-07-24 20:41:37                    2018-08-13           Early   
2      2018-08-08 08:38:49                    2018-09-04           Early   
3      2017-11-18 19:28:06                    2017-12-15           Early   
4      2018-02-13 21:18:39                    2018-02-26           Early   

   is_late  
0        0  
1        0  
2        0  
3        0  
4        0  
Dataset shape: (96476, 6)


In [20]:
# Count missing values in each column
print("Missing values:")
print(df.isna().sum())

# Check whether any order_id appears more than once
print("\nDuplicate orders:", df["order_id"].duplicated().sum())

# Inspect the data types of our columns
print("\nColumn data types:")
print(df.dtypes)


Missing values:
order_id                         0
customer_id                      0
order_purchase_timestamp         0
order_estimated_delivery_date    0
delivery_status                  0
is_late                          0
dtype: int64

Duplicate orders: 0

Column data types:
order_id                                    str
customer_id                                 str
order_purchase_timestamp         datetime64[us]
order_estimated_delivery_date    datetime64[us]
delivery_status                             str
is_late                                   int64
dtype: object


In [21]:
# Extract the month when the customer placed the order (1–12)
df["purchase_month"] = df["order_purchase_timestamp"].dt.month

# Extract the day of the week (Monday = 0, Sunday = 6)
df["purchase_day_of_week"] = df["order_purchase_timestamp"].dt.dayofweek

# Extract the hour when the order was placed (0–23)
df["purchase_hour"] = df["order_purchase_timestamp"].dt.hour

# Calculate the number of days between purchase and estimated delivery
df["estimated_delivery_days"] = (
    df["order_estimated_delivery_date"] - df["order_purchase_timestamp"]
).dt.total_seconds() / 86400 # converts seconds into days.

# Preview our new features alongside the prediction target
df[
    [
        "purchase_month",
        "purchase_day_of_week",
        "purchase_hour",
        "estimated_delivery_days",
        "is_late"
    ]
].head()


,purchase_month,purchase_day_of_week,purchase_hour,estimated_delivery_days,is_late
0,10,0,10,15.544063,0
1,7,1,20,19.137766,0
2,8,2,8,26.639711,0
3,11,5,19,26.188819,0
4,2,1,21,12.112049,0


In [22]:
# Retrieve customer IDs and their corresponding states from PostgreSQL
customers_query = """
SELECT
    customer_id,
    customer_state
FROM customers
"""

# Load the customer data into a separate DataFrame
customers_df = pd.read_sql(customers_query, engine)

# Merge customer states into our ML dataset using customer_id
# how="left" keeps every order from our original dataset
# validate="many_to_one" checks that each customer_id matches at most one customer record
df = df.merge(
    customers_df,
    on="customer_id",
    how="left",
    validate="many_to_one"
)

# Check whether any orders are missing customer state information
print("Missing customer states:", df["customer_state"].isna().sum())

# Preview our new feature
df[["customer_id", "customer_state", "is_late"]].head()


Missing customer states: 0


,customer_id,customer_state,is_late
0,9ef432eb6251297304e76186b10a928d,SP,0
1,b0830fb4747a6c6d20dea0b8c802d7ef,BA,0
2,41ce2a54c0b03bf3443c3d931a367089,GO,0
3,f88197465ea7920adcdbec7375364d82,RN,0
4,8ab97904e6daea8866dbdbc4fb7aad2c,SP,0


In [23]:
# Close all idle database connections managed by the engine
engine.dispose()